# ProgressionPred: Exploratory Data Analysis

This notebook performs exploratory data analysis on chronic disease progression data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Loading and Overview

In [ ]:
# Generate synthetic data for analysis
import sys
sys.path.append('..')
from train_progression_models import generate_synthetic_data

# Load data for each disease
diseases = ['Heart Disease', 'Type 2 Diabetes', 'Chronic Kidney Disease']
data = {}

for disease in diseases:
    data[disease] = generate_synthetic_data(disease, n_samples=1000)
    print(f"{disease} dataset shape: {data[disease].shape}")

In [ ]:
# Display basic statistics for Heart Disease
heart_data = data['Heart Disease']
print("Heart Disease Dataset Overview:")
print(heart_data.info())
print("\nBasic Statistics:")
heart_data.describe()

## 2. Disease Stage Distribution

In [ ]:
# Plot disease stage distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, disease in enumerate(diseases):
    stage_counts = data[disease]['current_stage'].value_counts().sort_index()
    axes[i].bar(stage_counts.index, stage_counts.values, alpha=0.7)
    axes[i].set_title(f'{disease}\nStage Distribution')
    axes[i].set_xlabel('Disease Stage')
    axes[i].set_ylabel('Count')
    axes[i].set_xticks([0, 1, 2, 3])
    axes[i].set_xticklabels(['Stage I', 'Stage II', 'Stage III', 'Stage IV'])

plt.tight_layout()
plt.show()

## 3. Biomarker Analysis

In [ ]:
# Heart Disease biomarker distributions by stage
heart_biomarkers = ['blood_pressure_systolic', 'blood_pressure_diastolic', 
                   'heart_rate', 'ejection_fraction']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, biomarker in enumerate(heart_biomarkers):
    sns.boxplot(data=heart_data, x='current_stage', y=biomarker, ax=axes[i])
    axes[i].set_title(f'{biomarker.replace("_", " ").title()} by Disease Stage')
    axes[i].set_xlabel('Disease Stage')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation heatmap for Heart Disease
numeric_cols = heart_data.select_dtypes(include=[np.number]).columns
correlation_matrix = heart_data[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Heart Disease Features Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Risk Factor Analysis

In [ ]:
# Analyze key risk factors
risk_factors = ['age', 'duration_years', 'compliance', 'exercise_hours', 
               'smoking_status', 'hypertension', 'diabetes']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, factor in enumerate(risk_factors):
    if factor in ['smoking_status', 'hypertension', 'diabetes']:
        # Categorical variables
        stage_by_factor = heart_data.groupby([factor, 'current_stage']).size().unstack(fill_value=0)
        stage_by_factor.plot(kind='bar', ax=axes[i], stacked=True)
        axes[i].set_title(f'Disease Stage by {factor.replace("_", " ").title()}')
        axes[i].legend(title='Stage', labels=['Stage I', 'Stage II', 'Stage III', 'Stage IV'])
    else:
        # Continuous variables
        sns.boxplot(data=heart_data, x='current_stage', y=factor, ax=axes[i])
        axes[i].set_title(f'{factor.replace("_", " ").title()} by Disease Stage')
    
    axes[i].tick_params(axis='x', rotation=45)

# Remove empty subplot
fig.delaxes(axes[7])
plt.tight_layout()
plt.show()

## 6. Progression Analysis

In [ ]:
# Progression patterns
progression_labels = ['Stable', 'Next Stage', 'Advanced']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, disease in enumerate(diseases):
    progression_counts = data[disease]['progression_12months'].value_counts().sort_index()
    axes[i].pie(progression_counts.values, labels=progression_labels, autopct='%1.1f%%')
    axes[i].set_title(f'{disease}\n12-Month Progression')

plt.tight_layout()
plt.show()

## 7. Complication Risk Analysis

In [ ]:
# Complication rates by disease stage
complications = ['complication_1', 'complication_2', 'complication_3']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, comp in enumerate(complications):
    comp_by_stage = heart_data.groupby('current_stage')[comp].mean()
    axes[i].bar(comp_by_stage.index, comp_by_stage.values, alpha=0.7)
    axes[i].set_title(f'Complication {i+1} Risk by Stage')
    axes[i].set_xlabel('Disease Stage')
    axes[i].set_ylabel('Risk Probability')
    axes[i].set_xticks([0, 1, 2, 3])
    axes[i].set_xticklabels(['Stage I', 'Stage II', 'Stage III', 'Stage IV'])

plt.tight_layout()
plt.show()

## 8. Interactive Visualizations

In [ ]:
# Interactive scatter plot
fig = px.scatter(heart_data, 
                x='age', 
                y='ejection_fraction',
                color='current_stage',
                size='duration_years',
                hover_data=['compliance', 'exercise_hours'],
                title='Heart Disease: Age vs Ejection Fraction by Stage',
                labels={'current_stage': 'Disease Stage'})

fig.show()

## 9. Key Insights Summary

In [ ]:
# Generate summary statistics
print("KEY INSIGHTS FROM EDA:")
print("=" * 50)

for disease in diseases:
    df = data[disease]
    print(f"\n{disease}:")
    print(f"  - Average age: {df['age'].mean():.1f} years")
    print(f"  - Average disease duration: {df['duration_years'].mean():.1f} years")
    print(f"  - Average compliance: {df['compliance'].mean():.1%}")
    print(f"  - Stage distribution: {df['current_stage'].value_counts().to_dict()}")
    print(f"  - Progression risk: {(df['progression_12months'] > 0).mean():.1%}")
    print(f"  - High-risk patients (Stage III+): {(df['current_stage'] >= 2).mean():.1%}")

## 10. Data Quality Assessment

In [ ]:
# Check for missing values and outliers
for disease in diseases:
    df = data[disease]
    print(f"\n{disease} Data Quality:")
    print(f"  - Missing values: {df.isnull().sum().sum()}")
    print(f"  - Duplicate rows: {df.duplicated().sum()}")
    print(f"  - Data types: {df.dtypes.value_counts().to_dict()}")
    
    # Check for outliers in key numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outliers = {}
    for col in numeric_cols[:5]:  # Check first 5 numeric columns
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers[col] = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    
    print(f"  - Outliers detected: {outliers}")